**Mini Project**

Airline Tweet Sentiment Classifier using Natural Language Processing**

Notes: Use sample dataset: https://github.com/salman1256/aimltraining/blob/main/Day-30/airline tweets sample.csv

Steps:

1. Import libraries

2. Load and explore dataset

3. Clean and preprocess the text

4. Convert text to numerical vectors (TF-IDF)

5. Split into train and test sets

6. Train a Logistic Regression model

7. Evaluate accuracy and classification report

8. Predict sentiment for new example tweets

In [10]:
# Steps 1a: Import libraries
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [11]:
# Steps 1b: Download nltk required thing like stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:
# Steps 2a: Load dataset
df = pd.read_csv("/content/sample_data/airline_tweets_sample.csv")

In [14]:
# Steps 2: Check data set at least top 5 values
df.head()


,text,sentiment
0,"@United flight was delayed for 3 hours, worst ...",negative
1,"Loved the service on @Delta, crew was super fr...",positive
2,"@AmericanAir lost my luggage again, so disappo...",negative
3,Smooth boarding and on-time arrival. Great job...,positive
4,The seats were uncomfortable but staff was polite,neutral


In [16]:
# Step 3
# Text cleaning and preprocessing
# for each tweet do:
# a. convert to lowercase
# b. remove urls
# c. remove special character and numbers
# d. remove stopwords (common words like *the, is and * etc)
# e. apply ** stemming** (reduce words to their root form)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+","",text)
    text = re.sub(r"[^a-zA-Z\s]","",text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [ps.stem(w) for w in words]
    return " ".join(words)

df["clean_tweet"] = df["text"].apply(clean_text)
df.head()


,text,sentiment,clean_tweet
0,"@United flight was delayed for 3 hours, worst ...",negative,unit flight delay hour worst experi ever
1,"Loved the service on @Delta, crew was super fr...",positive,love servic delta crew super friendli
2,"@AmericanAir lost my luggage again, so disappo...",negative,americanair lost luggag disappoint
3,Smooth boarding and on-time arrival. Great job...,positive,smooth board ontim arriv great job southwestair
4,The seats were uncomfortable but staff was polite,neutral,seat uncomfort staff polit


In [18]:
#STEP 4
#a) convert text to numerical vectors(TF-IDF)
#b) check X,y and shape len

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df["clean_tweet"])
y = df["sentiment"]

print("X shape:", X.shape)
print("y length:", len(y))


X shape: (30, 105)
y length: 30


In [19]:
# step 5: split into train and test sets
# 80% training AND 20% TESTING

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [24]:
# step 6: train a logistic regression model
# a: create logostic model
#b: train logistic model

model = LogisticRegression(max_iter=200, class_weight='balanced')
model.fit(X_train, y_train)


LogisticRegression(class_weight='balanced', max_iter=200)

In [25]:
#step 7:
# evaluate accuracy and classification report
# a. predict model
# b. precision, recall, F1-Score for each sentiments

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.3333333333333333

Classification Report:

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         1
     neutral       0.00      0.00      0.00         1
    positive       0.50      0.50      0.50         4

    accuracy                           0.33         6
   macro avg       0.17      0.17      0.17         6
weighted avg       0.33      0.33      0.33         6



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [26]:
# step 8: predict sentiment for new example tweets
new_tweets = [
    "The staff is so helpful!",
    "I am so disappointed. The flight got delayed by 5 hours.",
    "Nothing special. Just an average experience.",
    "The service was terrible and rude."
]

clean_new = [clean_text(t) for t in new_tweets]

new_vec = tfidf.transform(clean_new)

predictions = model.predict(new_vec)

for txt, pred in zip(new_tweets, predictions):
    print(f"{txt} ---> {pred}")


The staff is so helpful! ---> neutral
I am so disappointed. The flight got delayed by 5 hours. ---> negative
Nothing special. Just an average experience. ---> neutral
The service was terrible and rude. ---> negative
